# Suite TS5PRT — Printing TypeScript 5 programs from the store

`Programs.TypeScript5.Print` prints a program as the store holds it: a reference object that
`Programs.Framework.TheWorkspace` owns by its path, holding its tree as one value, its `body`. Each statement is a
value object, a branch of the union `Programs.TypeScript5.Statement` tagged by its kind in lower case, holding its
blocks, clauses, class members and parts as its properties, in their containers' order; its expressions are
mbse-expressions' TypeScript family's terms, which `TypeScript5.Text.Printer` writes and `Print.Printer` extends: where
a function, an arrow or a class expression holds statements (its `block` or `classBody`), they print within the term,
one level in from the statement that holds it, to any depth. The layout is the 0.3 printer's: four spaces a level,
braces on the line that opens them, `} else {` and `else if`, a statement that is no block on the next line one level
in, a semicolon after each statement that takes one, comments as written, a trailing one on the line it ends.

Each case builds its program in a transaction of its own, its tree written in the builders' own syntax: the program's
`body` a list of the union's branches, a union's branch by its tag (`lambda u: u.ifstatement(...)`), the terms nested
in it as values of mbse-expressions' unions (`a` below writes the identifier `a`). It checks the printed text. The
cases are ported from 0.3's suite TSPRT (`12_TypeScriptPrint.ipynb`), its expected texts kept, where the layout is
unchanged; its cases of expressions alone (precedence, types, each expression kind) are mbse-expressions' own suite's,
the expressions' printer being its. The text is not parsed back here: reading TypeScript 5 into the store is a module
of its own. The case setup is below.

In [ ]:
from mbse.Expressions.Framework import Objects, Terms
from mbse.Programs import TypeScript5 as TS5
from mbse.Programs.Framework import Model
from mbse.Programs.TypeScript5 import Print, Statements
from support import raises

store = TS5.store()
R = store.registered


def build(path, body):
    """The program at `path`, built in a transaction of its own: `body` its statements' writers, the builders'."""
    store.transaction(lambda t: t.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"ws": R("Programs.Framework.Workspace"), "p": R("Programs.TypeScript5.Program")})
        .spec([lambda _: _.ws.files({path: lambda e: e.file(_.p)}), lambda _: _.p.body(body)])]))]))


def printed(path, body):
    """The text of the program built at `path`."""
    build(path, body)
    return Print.print_module(store, path)


# The writers the cases share, in the builders' syntax: three names and a type, terms; the statements `b;` and `c;`
a = lambda t: t.identifier(lambda i: i.name("a"))  # noqa: E731
b = lambda t: t.identifier(lambda i: i.name("b"))  # noqa: E731
c = lambda t: t.identifier(lambda i: i.name("c"))  # noqa: E731
T = lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("T"))))  # noqa: E731
say_b = lambda u: u.expressionstatement(lambda s: s.expression(b))  # noqa: E731
say_c = lambda u: u.expressionstatement(lambda s: s.expression(c))  # noqa: E731

<a id="TS5PRT-01"></a>
## TS5PRT-01 · Every simple statement kind prints alone, on its line

A comment, `//` and `/* */`, its text after its delimiters; an expression statement, `;`, `debugger`, `return` with an
argument and without, `throw`, `break` and `continue` with a label and without; variable declarations with their
declarators (several, `declare`, a definite one, `x!: T`, a pattern); a type alias, with its type parameters and
`declare`; `export =` and `export as namespace`.

In [ ]:
text = printed("simple.ts", [
    lambda u: u.comment(lambda s: s.text(" a note")),
    lambda u: u.comment(lambda s: s.block(True).text(" a block ")),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.callexpression(lambda x: x.callee(a).arguments([b])))),
    lambda u: u.emptystatement(lambda s: s),
    lambda u: u.debuggerstatement(lambda s: s),
    lambda u: u.returnstatement(lambda s: s),
    lambda u: u.returnstatement(lambda s: s.argument(lambda e: e.sequenceexpression(lambda q: q.expressions([a, b])))),
    lambda u: u.throwstatement(lambda s: s.argument(a)),
    lambda u: u.breakstatement(lambda s: s),
    lambda u: u.breakstatement(lambda s: s.label(lambda i: i.name("outer"))),
    lambda u: u.continuestatement(lambda s: s),
    lambda u: u.continuestatement(lambda s: s.label(lambda i: i.name("outer"))),
    lambda u: u.variabledeclaration(lambda s: s.declarationKind("let").declarations([
        lambda d: d.id(a).init(lambda e: e.literal(lambda l: l.value(1.0))),
        lambda d: d.id(lambda p: p.identifier(lambda i: i.name("b").typeAnnotation(lambda n: n.typeAnnotation(T))))])),
    lambda u: u.variabledeclaration(lambda s: s.declare(True).declarationKind("var").declarations([lambda d: d.id(a)])),
    lambda u: u.variabledeclaration(lambda s: s.declarationKind("let").declarations([
        lambda d: d.id(lambda p: p.identifier(lambda i: i.name("x").typeAnnotation(lambda n: n.typeAnnotation(T))))
        .definite(True)])),
    lambda u: u.variabledeclaration(lambda s: s.declarationKind("const").declarations([
        lambda d: d.id(lambda p: p.objectpattern(lambda o: o.properties([lambda q: q.property(lambda r: r
            .propertyKind("init").key(a).value(a).shorthand(True))]))).init(b)])),
    lambda u: u.tstypealiasdeclaration(lambda s: s.id(lambda i: i.name("X")).typeAnnotation(T)),
    lambda u: u.tstypealiasdeclaration(lambda s: s.declare(True).id(lambda i: i.name("Pair"))
        .typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))
        .typeAnnotation(lambda t: t.tstupletype(lambda u: u.elementTypes([T, T])))),
    lambda u: u.tsexportassignment(lambda s: s.expression(a)),
    lambda u: u.tsnamespaceexportdeclaration(lambda s: s.id(lambda i: i.name("N"))),
])
assert text == """\
// a note
/* a block */
a(b);
;
debugger;
return;
return a, b;
throw a;
break;
break outer;
continue;
continue outer;
let a = 1, b: T;
declare var a;
let x!: T;
const { a } = b;
type X = T;
declare type Pair<T> = [T, T];
export = a;
export as namespace N;
"""

<a id="TS5PRT-02"></a>
## TS5PRT-02 · Every compound statement kind prints, with its clauses

A block, empty and not; `if` and `else`; `while` and `do ... while`; `for` with a declaration, an expression or nothing
as its initializer, and its test and update; `for ... in` and `for ... of`, its left side a declaration or a pattern,
`for await`; a labeled statement; `with`; `switch` and its cases, `case` and `default`, a trailing comment first in a
case on its line; `try` with its `catch` clause, with a parameter and without, and `finally`.

In [ ]:
text = printed("compound.ts", [
    lambda u: u.blockstatement(lambda s: s.body([say_b])),
    lambda u: u.blockstatement(lambda s: s),
    lambda u: u.ifstatement(lambda s: s.test(a).consequent(lambda c_: c_.blockstatement(lambda k: k.body([say_b])))
                            .alternate(lambda e: e.blockstatement(lambda k: k.body([say_b])))),
    lambda u: u.whilestatement(lambda s: s.test(a).body(say_b)),
    lambda u: u.dowhilestatement(lambda s: s.body(lambda k: k.blockstatement(lambda k: k.body([say_b]))).test(a)),
    lambda u: u.forstatement(lambda s: s.init(lambda i: i.variabledeclaration(lambda d: d.declarationKind("let")
        .declarations([lambda d: d.id(lambda p: p.identifier(lambda i: i.name("i"))).init(lambda e: e.literal(lambda l: l.value(0.0)))])))
        .test(lambda e: e.binaryexpression(lambda x: x.left(lambda t: t.identifier(lambda i: i.name("i"))).operator("<").right(a)))
        .update(lambda e: e.updateexpression(lambda x: x.operator("++").argument(lambda t: t.identifier(lambda i: i.name("i")))))
        .body(lambda k: k.blockstatement(lambda k: k.body([say_b])))),
    lambda u: u.forstatement(lambda s: s.init(lambda i: i.expression(lambda e: e.assignmentexpression(lambda x: x.left(a).operator("=").right(b))))
        .body(lambda k: k.emptystatement(lambda e: e))),
    lambda u: u.forstatement(lambda s: s.body(lambda k: k.blockstatement(lambda k: k))),
    lambda u: u.forinstatement(lambda s: s.left(lambda l: l.variabledeclaration(lambda d: d.declarationKind("const")
        .declarations([lambda d: d.id(lambda p: p.identifier(lambda i: i.name("k")))]))).right(a).body(say_b)),
    lambda u: u.forofstatement(lambda s: s.left(lambda l: l.pattern(lambda p: p.arraypattern(lambda r: r.elements([a, b]))))
        .right(c).body(lambda k: k.blockstatement(lambda k: k.body([say_b])))),
    lambda u: u.forofstatement(lambda s: s.isAwait(True).left(lambda l: l.variabledeclaration(lambda d: d.declarationKind("const")
        .declarations([lambda d: d.id(a)]))).right(c).body(lambda k: k.blockstatement(lambda k: k))),
    lambda u: u.labeledstatement(lambda s: s.label(lambda i: i.name("outer")).body(lambda k: k.forofstatement(lambda f: f
        .left(lambda l: l.pattern(a)).right(b).body(lambda k: k.continuestatement(lambda s: s.label(lambda i: i.name("outer"))))))),
    lambda u: u.withstatement(lambda s: s.object(a).body(lambda k: k.blockstatement(lambda k: k.body([say_b])))),
    lambda u: u.switchstatement(lambda s: s.discriminant(a).cases([
        lambda k: k.switchcase(lambda w: w.test(lambda e: e.literal(lambda l: l.value(1.0))).consequent([
            lambda u: u.comment(lambda s: s.text(" one").trailing(True)), say_b,
            lambda u: u.breakstatement(lambda s: s)])),
        lambda k: k.switchcase(lambda w: w.test(lambda e: e.literal(lambda l: l.text("'two'")))),
        lambda k: k.switchcase(lambda w: w.consequent([lambda u: u.returnstatement(lambda s: s)]))])),
    lambda u: u.trystatement(lambda s: s.block(lambda k: k.body([say_b]))
        .handler(lambda h: h.catchclause(lambda w: w.param(lambda p: p.identifier(lambda i: i.name("e")))
                                         .body(lambda k: k.body([say_b]))))
        .finalizer(lambda k: k.body([say_b]))),
    lambda u: u.trystatement(lambda s: s.block(lambda k: k).handler(lambda h: h.catchclause(lambda w: w.body(lambda k: k)))),
])
assert text == """\
{
    b;
}
{}
if (a) {
    b;
} else {
    b;
}
while (a)
    b;
do {
    b;
} while (a);
for (let i = 0; i < a; i++) {
    b;
}
for (a = b;;)
    ;
for (;;) {}
for (const k in a)
    b;
for ([a, b] of c) {
    b;
}
for await (const a of c) {}
outer: for (a of b)
    continue outer;
with (a) {
    b;
}
switch (a) {
    case 1: // one
        b;
        break;
    case 'two':
    default:
        return;
}
try {
    b;
} catch (e) {
    b;
} finally {
    b;
}
try {} catch {}
"""

<a id="TS5PRT-03"></a>
## TS5PRT-03 · Declarations: functions, classes and every class member, interfaces, enums and modules

A function declaration with `async`, `*`, type parameters, parameters of every pattern and a return type; one with no
body (`TSDeclareFunction`), `declare`d and not. A class with decorators, `abstract`, type parameters, a superclass and
its type arguments and `implements`, and every member: fields (`PropertyDefinition`, `TSAbstractPropertyDefinition`,
`AccessorProperty`, `TSAbstractAccessorProperty`) with each modifier in TypeScript's order, `declare`, `!`, `?`, a
computed key, a type and a value; a static block; an index signature; comments; methods (`MethodDefinition`,
`TSAbstractMethodDefinition`): a constructor with a parameter property, a getter, a setter, a decorated
`static override async *[b]?()`, and methods without a body, each with its statements where it has a block.

In [ ]:
number = lambda t: t.typeAnnotation(lambda k: k.tsnumberkeyword(lambda n: n))  # noqa: E731
functions = printed("functions.ts", [
    lambda u: u.functiondeclaration(lambda f: f.isAsync(True).generator(True).id(lambda i: i.name("f"))
        .typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T")).constraint(T)]))
        .params([lambda p: p.identifier(lambda i: i.name("x").typeAnnotation(lambda n: n.typeAnnotation(T))),
                 lambda p: p.assignmentpattern(lambda q: q.left(b).right(lambda e: e.literal(lambda l: l.value(1.0)))),
                 lambda p: p.restelement(lambda r: r.argument(c))])
        .returnType(number).body(lambda k: k.body([say_b]))),
    lambda u: u.functiondeclaration(lambda f: f.id(lambda i: i.name("g")).body(lambda k: k)),
    lambda u: u.tsdeclarefunction(lambda f: f.declare(True).id(lambda i: i.name("h")).params([a]).returnType(number)),
    lambda u: u.tsdeclarefunction(lambda f: f.id(lambda i: i.name("h"))),
])
assert functions == """\
async function* f<T extends T>(x: T, b = 1, ...c): number {
    b;
}
function g() {}
declare function h(a): number;
function h();
"""

In [ ]:
classes = printed("classes.ts", [
    lambda u: u.classdeclaration(lambda k: k.decorators([lambda d: d.expression(lambda e: e.callexpression(lambda x: x.callee(a)))])
        .declare(False).abstract(True).id(lambda i: i.name("C"))
        .typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))
        .superClass(b).superTypeArguments(lambda s: s.params([T]))
        .implements([lambda i: i.expression(c), lambda i: i.expression(a).typeArguments(lambda s: s.params([T]))])
        .body(lambda y: y.body([
            lambda m: m.comment(lambda s: s.text(" on the head").trailing(True)),
            lambda m: m.propertydefinition(lambda p: p.decorators([lambda d: d.expression(a)]).accessibility("private")
                                           .static(True).readonly(True).key(a).typeAnnotation(number)
                                           .value(lambda e: e.literal(lambda l: l.value(1.0)))),
            lambda m: m.propertydefinition(lambda p: p.declare(True).key(b).definite(True).typeAnnotation(number)),
            lambda m: m.propertydefinition(lambda p: p.computed(True).key(c).optional(True)),
            lambda m: m.tsabstractpropertydefinition(lambda p: p.accessibility("protected").key(a).typeAnnotation(number)),
            lambda m: m.accessorproperty(lambda p: p.override(True).key(a)),
            lambda m: m.tsabstractaccessorproperty(lambda p: p.key(b).typeAnnotation(number)),
            lambda m: m.tsindexsignature(lambda s: s.static(True).parameters([lambda p: p.identifier(lambda i: i.name("k")
                .typeAnnotation(lambda n: n.typeAnnotation(lambda k: k.tsstringkeyword(lambda n: n))))]).typeAnnotation(number)),
            lambda m: m.staticblock(lambda s: s.body([say_b])),
            lambda m: m.comment(lambda s: s.text(" methods")),
            lambda m: m.methoddefinition(lambda d: d.methodKind("constructor").key(lambda t: t.identifier(lambda i: i.name("constructor")))
                .value(lambda f: f.functionexpression(lambda f: f.params([lambda p: p.tsparameterproperty(lambda q: q
                    .accessibility("public").readonly(True).parameter(a))]).block(lambda k: k.blockstatement(lambda k: k.body([say_b])))))),
            lambda m: m.methoddefinition(lambda d: d.methodKind("get").key(a).value(lambda f: f.functionexpression(lambda f: f
                .returnType(number).block(lambda k: k.blockstatement(lambda k: k.body([
                    lambda u: u.returnstatement(lambda s: s.argument(lambda e: e.literal(lambda l: l.value(1.0))))])))))),
            lambda m: m.methoddefinition(lambda d: d.methodKind("set").key(a).value(lambda f: f.functionexpression(lambda f: f
                .params([b]).block(lambda k: k.blockstatement(lambda k: k))))),
            lambda m: m.methoddefinition(lambda d: d.decorators([lambda d: d.expression(c)]).accessibility("public")
                .static(True).override(True).methodKind("method").computed(True).key(b).optional(True)
                .value(lambda f: f.functionexpression(lambda f: f.isAsync(True).generator(True).block(lambda k: k.blockstatement(lambda k: k))))),
            lambda m: m.methoddefinition(lambda d: d.methodKind("method").key(c).value(lambda f: f.tsemptybodyfunctionexpression(lambda f: f))),
            lambda m: m.tsabstractmethoddefinition(lambda d: d.accessibility("protected").methodKind("get").key(c)
                .value(lambda f: f.tsemptybodyfunctionexpression(lambda f: f.returnType(number)))),
        ]))),
    lambda u: u.classdeclaration(lambda k: k.declare(True).id(lambda i: i.name("D")).body(lambda y: y)),
])
assert classes == """\
@a()
abstract class C<T> extends b<T> implements c, a<T> { // on the head
    @a
    private static readonly a: number = 1;
    declare b!: number;
    [c]?;
    protected abstract a: number;
    override accessor a;
    abstract accessor b: number;
    static [k: string]: number;
    static {
        b;
    }
    // methods
    constructor(public readonly a) {
        b;
    }
    get a(): number {
        return 1;
    }
    set a(b) {}
    @c
    public static override async *[b]?() {}
    c();
    protected abstract get c(): number;
}
declare class D {}
"""

An interface with its type parameters, its heritage and each of `TypeScript Types`' members, comments among them; an
enum, `declare` and `const`, its members one a line each with its comma, an initializer, a computed key and comments
among them, an enum with an empty body and one with none; modules: a namespace, a module named by a string, a
namespace named by a qualified name, `declare global`, and a module with no body.

In [ ]:
types = printed("types.ts", [
    lambda u: u.tsinterfacedeclaration(lambda s: s.declare(True).id(lambda i: i.name("I"))
        .typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))
        .extends([lambda h: h.expression(a), lambda h: h.expression(lambda e: e.memberexpression(lambda m: m.object(a).property(b)))
                  .typeArguments(lambda s: s.params([T]))])
        .body(lambda y: y.body([
            lambda m: m.tspropertysignature(lambda p: p.readonly(True).key(a).optional(True).typeAnnotation(number)),
            lambda m: m.comment(lambda s: s.text(" a")),
            lambda m: m.tsmethodsignature(lambda p: p.methodKind("method").key(b).params([c]).returnType(number)),
            lambda m: m.comment(lambda s: s.block(True).text(" b ").trailing(True)),
            lambda m: m.tscallsignaturedeclaration(lambda p: p.params([a])),
            lambda m: m.tsconstructsignaturedeclaration(lambda p: p.returnType(lambda n: n.typeAnnotation(T))),
            lambda m: m.tsindexsignature(lambda p: p.readonly(True).parameters([lambda p: p.identifier(lambda i: i.name("k")
                .typeAnnotation(lambda n: n.typeAnnotation(lambda k: k.tsstringkeyword(lambda n: n))))]).typeAnnotation(number)),
        ]))),
    lambda u: u.tsinterfacedeclaration(lambda s: s.id(lambda i: i.name("J")).body(lambda y: y)),
    lambda u: u.tsenumdeclaration(lambda s: s.declare(True).const(True).id(lambda i: i.name("E")).body(lambda y: y.members([
        lambda m: m.comment(lambda s: s.text(" first").trailing(True)),
        lambda m: m.tsenummember(lambda e: e.id(a)),
        lambda m: m.tsenummember(lambda e: e.id(lambda t: t.literal(lambda l: l.text("'b'"))).initializer(lambda e: e.literal(lambda l: l.value(2.0)))),
        lambda m: m.comment(lambda s: s.text(" computed")),
        lambda m: m.tsenummember(lambda e: e.computed(True).id(c)),
    ]))),
    lambda u: u.tsenumdeclaration(lambda s: s.id(lambda i: i.name("F")).body(lambda y: y)),
    lambda u: u.tsenumdeclaration(lambda s: s.id(lambda i: i.name("G"))),
    lambda u: u.tsmoduledeclaration(lambda s: s.moduleKind("namespace").id(lambda n: n.identifier(lambda i: i.name("N")))
        .body(lambda y: y.body([say_b]))),
    lambda u: u.tsmoduledeclaration(lambda s: s.declare(True).moduleKind("module").id(lambda n: n.literal(lambda l: l.text("'m'")))
        .body(lambda y: y)),
    lambda u: u.tsmoduledeclaration(lambda s: s.moduleKind("namespace").id(lambda n: n.tsqualifiedname(lambda q: q.left(a)
        .right(lambda i: i.name("b")))).body(lambda y: y.body([say_b]))),
    lambda u: u.tsmoduledeclaration(lambda s: s.declare(True).moduleKind("global").id(lambda n: n.identifier(lambda i: i.name("global")))
        .body(lambda y: y)),
    lambda u: u.tsmoduledeclaration(lambda s: s.declare(True).moduleKind("module").id(lambda n: n.literal(lambda l: l.text("'n'")))),
])
assert types == """\
declare interface I<T> extends a, a.b<T> {
    readonly a?: number;
    // a
    b(c): number; /* b */
    (a);
    new (): T;
    readonly [k: string]: number;
}
interface J {}
declare const enum E { // first
    a,
    'b' = 2,
    // computed
    [c],
}
enum F {}
enum G {}
namespace N {
    b;
}
declare module 'm' {}
namespace a.b {
    b;
}
declare global {}
declare module 'n';
"""

<a id="TS5PRT-04"></a>
## TS5PRT-04 · Modules: imports and exports, with every specifier and attribute

`import` for its effect, of a default, a namespace, named specifiers (`as`, `type`), `import type`, `import defer`,
with attributes; `export` of a declaration (decorators before it), of specifiers, `export type`, re-exports with
attributes, `export {}`; `export default` of a declaration and of an expression; `export *`, `export * as`,
`export type *`; `import ... =` an identifier, a qualified name and `require(...)`.

In [ ]:
m = lambda l: l.text("'m'")  # noqa: E731
json = lambda a_: a_.key(lambda t: t.identifier(lambda i: i.name("type"))).value(lambda l: l.text("'json'"))  # noqa: E731
modules = printed("modules.ts", [
    lambda u: u.importdeclaration(lambda s: s.source(m)),
    lambda u: u.importdeclaration(lambda s: s.specifiers([lambda c_: c_.importdefaultspecifier(lambda p: p.local(lambda i: i.name("d")))]).source(m)),
    lambda u: u.importdeclaration(lambda s: s.importKind("type").specifiers([
        lambda c_: c_.importdefaultspecifier(lambda p: p.local(lambda i: i.name("a"))),
        lambda c_: c_.importnamespacespecifier(lambda p: p.local(lambda i: i.name("b")))]).source(m)),
    lambda u: u.importdeclaration(lambda s: s.specifiers([
        lambda c_: c_.importspecifier(lambda p: p.imported(a).local(lambda i: i.name("a"))),
        lambda c_: c_.importspecifier(lambda p: p.importKind("type").imported(b).local(lambda i: i.name("c"))),
        lambda c_: c_.importdefaultspecifier(lambda p: p.local(lambda i: i.name("d")))]).source(m).attributes([json])),
    lambda u: u.importdeclaration(lambda s: s.phase("defer").specifiers([
        lambda c_: c_.importnamespacespecifier(lambda p: p.local(lambda i: i.name("ns")))]).source(m)),
    lambda u: u.exportnameddeclaration(lambda s: s.declaration(lambda d: d.variabledeclaration(lambda v: v.declarationKind("const")
        .declarations([lambda d: d.id(a).init(b)])))),
    lambda u: u.exportnameddeclaration(lambda s: s.declaration(lambda d: d.classdeclaration(lambda k: k
        .decorators([lambda d: d.expression(a)]).id(lambda i: i.name("C")).body(lambda y: y)))),
    lambda u: u.exportnameddeclaration(lambda s: s.specifiers([
        lambda p: p.local(a).exported(a), lambda p: p.exportKind("type").local(a).exported(b)])),
    lambda u: u.exportnameddeclaration(lambda s: s.exportKind("type").specifiers([lambda p: p.local(c).exported(c)])
        .source(m).attributes([json])),
    lambda u: u.exportnameddeclaration(lambda s: s),
    lambda u: u.exportdefaultdeclaration(lambda s: s.declaration(lambda d: d.statement(lambda t: t.functiondeclaration(lambda f: f
        .id(lambda i: i.name("f")).body(lambda k: k.body([say_b])))))),
    lambda u: u.exportdefaultdeclaration(lambda s: s.declaration(lambda d: d.expression(a))),
    lambda u: u.exportalldeclaration(lambda s: s.source(m)),
    lambda u: u.exportalldeclaration(lambda s: s.exported(a).source(m)),
    lambda u: u.exportalldeclaration(lambda s: s.exportKind("type").source(m).attributes([json])),
    lambda u: u.tsimportequalsdeclaration(lambda s: s.id(lambda i: i.name("a")).moduleReference(lambda r: r.identifier(lambda i: i.name("b")))),
    lambda u: u.tsimportequalsdeclaration(lambda s: s.importKind("type").id(lambda i: i.name("a"))
        .moduleReference(lambda r: r.tsqualifiedname(lambda q: q.left(b).right(lambda i: i.name("c"))))),
    lambda u: u.tsimportequalsdeclaration(lambda s: s.id(lambda i: i.name("fs"))
        .moduleReference(lambda r: r.tsexternalmodulereference(lambda x: x.expression(lambda l: l.text("'fs'"))))),
])
assert modules == """\
import 'm';
import d from 'm';
import type a, * as b from 'm';
import d, {a, type b as c} from 'm' with { type: 'json' };
import defer * as ns from 'm';
export const a = b;
@a
export class C {}
export {a, type a as b};
export type {c} from 'm' with { type: 'json' };
export {};
export default function f() {
    b;
}
export default a;
export * from 'm';
export * as a from 'm';
export type * from 'm' with { type: 'json' };
import a = b;
import type a = b.c;
import fs = require('fs');
"""

<a id="TS5PRT-05"></a>
## TS5PRT-05 · Statements within terms: functions, arrows, classes and object methods, nested

Where a term holds statements, they print at the level of the statement that holds the term, one level in, its closing
brace at that statement's level: 0.3's case, a class expression whose method has a block, returned from an arrow's
block, in a declaration in a function; and deeper, an object's method returning an arrow and its getter, in a function
expression passed within an `if`, in an async arrow passed to a call.

In [ ]:
# 0.3's case: a class expression returned from an arrow's block, in a declaration within a function
method_b = lambda m: m.methoddefinition(lambda d: d.methodKind("method").key(b).value(  # noqa: E731
    lambda f: f.functionexpression(lambda f: f.block(lambda k: k.blockstatement(lambda k: k.body([say_c]))))))
returns_class = lambda u: u.returnstatement(lambda s: s.argument(  # noqa: E731
    lambda e: e.classexpression(lambda x: x.classBody(lambda y: y.classbody(lambda y: y.body([method_b]))))))
arrow = lambda e: e.arrowfunctionexpression(  # noqa: E731
    lambda r: r.block(lambda k: k.blockstatement(lambda k: k.body([returns_class]))))
layout = printed("within.ts", [
    lambda u: u.functiondeclaration(lambda f: f.id(lambda i: i.name("f")).body(lambda k: k.body([
        lambda u: u.variabledeclaration(lambda s: s.declarationKind("const").declarations([lambda d: d.id(a).init(arrow)])),
        lambda u: u.comment(lambda s: s.text(" trailing").trailing(True))]))),
])
assert layout == """\
function f() {
    const a = () => {
        return class {
            b() {
                c;
            }
        };
    }; // trailing
}
"""

# Deeper: an object's method returning an arrow, in a function expression called within an `if`, in an async arrow,
# an argument of an expression statement
returns_arrow = lambda u: u.returnstatement(lambda s: s.argument(  # noqa: E731
    lambda e: e.arrowfunctionexpression(lambda r: r.block(lambda k: k.blockstatement(lambda k: k.body([say_c]))))))
getter = lambda u: u.returnstatement(lambda s: s.argument(a))  # noqa: E731
returns_object = lambda u: u.returnstatement(lambda s: s.argument(lambda e: e.objectexpression(lambda o: o.properties([  # noqa: E731
    lambda p: p.property(lambda q: q.propertyKind("init").key(a).value(b)),
    lambda p: p.property(lambda q: q.propertyKind("init").method(True).key(lambda t: t.identifier(lambda i: i.name("m")))
                         .value(lambda f: f.functionexpression(lambda f: f.block(
                             lambda k: k.blockstatement(lambda k: k.body([returns_arrow])))))),
    lambda p: p.property(lambda q: q.propertyKind("get").key(c).value(lambda f: f.functionexpression(lambda f: f.block(
        lambda k: k.blockstatement(lambda k: k.body([getter])))))),
]))))
calls_g = lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.callexpression(lambda x: x.callee(c).arguments([  # noqa: E731
    lambda t: t.functionexpression(lambda f: f.id(lambda i: i.name("g")).params([a]).block(
        lambda k: k.blockstatement(lambda k: k.body([returns_object]))))]))))
callback = lambda t: t.arrowfunctionexpression(lambda r: r.isAsync(True).params([b]).block(  # noqa: E731
    lambda k: k.blockstatement(lambda k: k.body([
        lambda u: u.ifstatement(lambda s: s.test(b).consequent(lambda k: k.blockstatement(lambda k: k.body([calls_g]))))]))))
deep = printed("deep.ts", [
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.callexpression(lambda x: x
        .callee(lambda t: t.memberexpression(lambda m: m.object(a).property(lambda t: t.identifier(lambda i: i.name("then")))))
        .arguments([callback])))),
])
assert deep == """\
a.then(async (b) => {
    if (b) {
        c(function g(a) {
            return { a: b, m() {
                return () => {
                    c;
                };
            }, get c() {
                return a;
            } };
        });
    }
});
"""

A function expression or a class expression that starts an expression statement is parenthesized with its block;
an arrow with an expression body, a block, or neither (`{}`), its lone type parameter `<T,>` as `Text.Printer` writes
it; an object's setter, its method without a body, a shorthand, and a property whose value is a function expression.

In [ ]:
starts = printed("starts.ts", [
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.callexpression(lambda x: x.callee(
        lambda t: t.functionexpression(lambda f: f.block(lambda k: k.blockstatement(lambda k: k.body([say_b])))))))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.classexpression(lambda x: x.id(lambda i: i.name("K"))
        .superClass(a).classBody(lambda y: y.classbody(lambda y: y.body([lambda m: m.propertydefinition(lambda p: p.key(b))])))))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.arrowfunctionexpression(lambda r: r.params([a]).body(b)))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.arrowfunctionexpression(lambda r: r
        .typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).block(lambda k: k.blockstatement(lambda k: k))))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.arrowfunctionexpression(lambda r: r))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.objectexpression(lambda o: o.properties([
        lambda p: p.property(lambda q: q.propertyKind("set").key(a).value(lambda f: f.functionexpression(lambda f: f.params([b])
            .block(lambda k: k.blockstatement(lambda k: k.body([say_c])))))),
        lambda p: p.property(lambda q: q.propertyKind("init").method(True).key(b).value(lambda f: f.tsemptybodyfunctionexpression(lambda f: f))),
        lambda p: p.property(lambda q: q.propertyKind("init").shorthand(True).key(c).value(c)),
        lambda p: p.property(lambda q: q.propertyKind("init").key(a).value(lambda f: f.functionexpression(lambda f: f
            .block(lambda k: k.blockstatement(lambda k: k.body([say_c])))))),
    ])))),
])
assert starts == """\
(function() {
    b;
}());
(class K extends a {
    b;
});
(a) => b;
<T,>() => {};
() => {};
({ set a(b) {
    c;
}, b(), c, a: function() {
    c;
} });
"""

<a id="TS5PRT-06"></a>
## TS5PRT-06 · Where an expression starts a statement, a `for` or a default export

0.3's cases: an expression statement that would start with `{`, `function`, `class` or `let [` is parenthesized; a
`for`'s initializer that holds `in` outside parentheses is, wherever within it; `for ((async) of a)` but
`for await (async of a)`; a `for ... of`'s sequence; `export default` of a function or class expression, and of a
sequence, is parenthesized, of an arrow not.

In [ ]:
empty_function = lambda t: t.functionexpression(lambda f: f.block(lambda k: k.blockstatement(lambda k: k)))  # noqa: E731
empty_class = lambda t: t.classexpression(lambda x: x.classBody(lambda y: y.classbody(lambda y: y)))  # noqa: E731
x_in_b = lambda t: t.binaryexpression(lambda x: x.left(lambda l: l.literal(lambda l: l.text("'x'"))).operator("in").right(b))  # noqa: E731
a_in_b = lambda t: t.binaryexpression(lambda x: x.left(a).operator("in").right(b))  # noqa: E731
empty = lambda k: k.emptystatement(lambda e: e)  # noqa: E731
precedence = printed("starts_statements.ts", [
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.objectexpression(lambda o: o))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.callexpression(lambda x: x.callee(empty_function)))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.memberexpression(lambda m: m.object(empty_class).property(a)))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.assignmentexpression(lambda x: x
        .left(lambda p: p.objectpattern(lambda o: o)).operator("=").right(a)))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.assignmentexpression(lambda x: x
        .left(lambda p: p.memberexpression(lambda m: m.object(lambda t: t.identifier(lambda i: i.name("let"))).property(a).computed(True)))
        .operator("=").right(b)))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.updateexpression(lambda x: x.operator("++")
        .argument(lambda t: t.memberexpression(lambda m: m.object(lambda o: o.objectexpression(lambda o: o)).property(a)))))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.tsasexpression(lambda x: x
        .expression(lambda o: o.objectexpression(lambda o: o)).typeAnnotation(T)))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.conditionalexpression(lambda x: x
        .test(lambda t: t.taggedtemplateexpression(lambda g: g.tag(empty_function).quasi(lambda q: q.quasis([lambda e: e.raw("").tail(True)]))))
        .consequent(a).alternate(b)))),
    lambda u: u.forstatement(lambda s: s.init(lambda i: i.variabledeclaration(lambda d: d.declarationKind("let")
        .declarations([lambda d: d.id(a).init(x_in_b)]))).body(empty)),
    lambda u: u.forstatement(lambda s: s.init(lambda i: i.expression(lambda e: e.binaryexpression(lambda x: x.left(a).operator("+").right(b))))
        .body(empty)),
    lambda u: u.forofstatement(lambda s: s.left(lambda l: l.pattern(lambda p: p.identifier(lambda i: i.name("async")))).right(a).body(empty)),
    lambda u: u.forofstatement(lambda s: s.isAwait(True).left(lambda l: l.pattern(lambda p: p.identifier(lambda i: i.name("async"))))
        .right(a).body(empty)),
    lambda u: u.forofstatement(lambda s: s.left(lambda l: l.pattern(a)).right(lambda e: e.sequenceexpression(lambda q: q.expressions([a, b])))
        .body(empty)),
    lambda u: u.forinstatement(lambda s: s.left(lambda l: l.pattern(lambda p: p.memberexpression(lambda m: m.object(a).property(b))))
        .right(c).body(lambda k: k.blockstatement(lambda k: k))),
    lambda u: u.exportdefaultdeclaration(lambda s: s.declaration(lambda d: d.expression(empty_function))),
    lambda u: u.exportdefaultdeclaration(lambda s: s.declaration(lambda d: d.expression(lambda e: e.callexpression(lambda x: x.callee(empty_class))))),
    lambda u: u.exportdefaultdeclaration(lambda s: s.declaration(lambda d: d.expression(lambda e: e.sequenceexpression(lambda q: q.expressions([a, b]))))),
    lambda u: u.exportdefaultdeclaration(lambda s: s.declaration(lambda d: d.expression(lambda e: e.arrowfunctionexpression(lambda r: r.body(a))))),
])
assert precedence == """\
({});
(function() {}());
(class {}.a);
({} = a);
(let[a] = b);
({}.a++);
({} as T);
(function() {}`` ? a : b);
for (let a = ('x' in b);;)
    ;
for (a + b;;)
    ;
for ((async) of a)
    ;
for await (async of a)
    ;
for (a of (a, b))
    ;
for (a.b in c) {}
export default (function() {});
export default (class {}());
export default (a, b);
export default () => a;
"""

In [ ]:
# A `for`'s initializer holding `in` outside parentheses, wherever within it, is parenthesized; a block holds none
inits = [
    lambda e: e.conditionalexpression(lambda x: x.test(a_in_b).consequent(a).alternate(b)),
    lambda e: e.sequenceexpression(lambda q: q.expressions([a, a_in_b])),
    lambda e: e.tsasexpression(lambda x: x.expression(a_in_b).typeAnnotation(T)),
    lambda e: e.arrowfunctionexpression(lambda r: r.body(a_in_b)),
    lambda e: e.logicalexpression(lambda x: x.left(a).operator("&&").right(a_in_b)),
    lambda e: e.assignmentexpression(lambda x: x.left(a).operator("=").right(a_in_b)),
    lambda e: e.conditionalexpression(lambda x: x.test(a).consequent(b).alternate(a_in_b)),
    lambda e: e.arrowfunctionexpression(lambda r: r.block(lambda k: k.blockstatement(lambda k: k))),
]
text = printed("inits.ts", [lambda u, init=init: u.forstatement(lambda s: s.init(lambda i: i.expression(init)).body(empty))
                            for init in inits])
assert text.split("\n")[:-1:2] == [
    "for ((a in b ? a : b);;)", "for ((a, a in b);;)", "for ((a in b as T);;)", "for ((() => a in b);;)",
    "for ((a && a in b);;)", "for ((a = a in b);;)", "for ((a ? b : a in b);;)", "for (() => {};;)"]

<a id="TS5PRT-07"></a>
## TS5PRT-07 · Layout: `else if`, blocks and statements, comments, an object type's lines, a hashbang

0.3's cases, their texts kept: `} else if (b)` and a statement that is no block on the next line, `do`'s `while` on
the line after its statement, labels before what they label; a trailing comment first in a block on its opening line;
an enum's comments; an object type with comments on its members' lines at the level of the statement that holds it,
and without on one line; a program's hashbang.

In [ ]:
say_d = lambda u: u.expressionstatement(lambda s: s.expression(lambda t: t.identifier(lambda i: i.name("d"))))  # noqa: E731
nested = printed("else.ts", [
    lambda u: u.ifstatement(lambda s: s.test(a).consequent(lambda k: k.blockstatement(lambda k: k.body([say_b])))
        .alternate(lambda e: e.ifstatement(lambda s: s.test(b).consequent(say_c).alternate(lambda e: e.blockstatement(lambda k: k.body([
            lambda u: u.ifstatement(lambda s: s.test(c).consequent(lambda k: k.blockstatement(lambda k: k)).alternate(say_d))])))))),
    lambda u: u.ifstatement(lambda s: s.test(a).consequent(say_b).alternate(lambda e: e.blockstatement(lambda k: k))),
    lambda u: u.ifstatement(lambda s: s.test(a).consequent(lambda k: k.blockstatement(lambda k: k))),
    lambda u: u.trystatement(lambda s: s.block(lambda k: k).finalizer(lambda k: k.body([say_b]))),
    lambda u: u.classdeclaration(lambda k: k.id(lambda i: i.name("C")).superClass(b).body(lambda y: y)),
    lambda u: u.dowhilestatement(lambda s: s.body(say_b).test(a)),
    lambda u: u.labeledstatement(lambda s: s.label(lambda i: i.name("a")).body(lambda k: k.labeledstatement(lambda s: s
        .label(lambda i: i.name("b")).body(lambda k: k.emptystatement(lambda e: e))))),
    lambda u: u.withstatement(lambda s: s.object(a).body(say_b)),
])
assert nested == """\
if (a) {
    b;
} else if (b)
    c;
else {
    if (c) {} else
        d;
}
if (a)
    b;
else {}
if (a) {}
try {} finally {
    b;
}
class C extends b {}
do
    b;
while (a);
a: b: ;
with (a)
    b;
"""

In [ ]:
comments = printed("comments.ts", [
    lambda u: u.blockstatement(lambda s: s.body([lambda u: u.comment(lambda s: s.text(" only").trailing(True))])),
    lambda u: u.blockstatement(lambda s: s.body([lambda u: u.comment(lambda s: s.block(True).text(" only ").trailing(True))])),
    lambda u: u.tsenumdeclaration(lambda s: s.id(lambda i: i.name("E")).body(lambda y: y.members([
        lambda m: m.comment(lambda s: s.text(" a").trailing(True)), lambda m: m.tsenummember(lambda e: e.id(a)),
        lambda m: m.comment(lambda s: s.text(" b")), lambda m: m.comment(lambda s: s.text(" c").trailing(True))]))),
    lambda u: u.tsmoduledeclaration(lambda s: s.moduleKind("namespace").id(lambda n: n.identifier(lambda i: i.name("N")))
        .body(lambda y: y.body([lambda u: u.tstypealiasdeclaration(lambda s: s.id(lambda i: i.name("X")).typeAnnotation(
            lambda t: t.tstypeliteral(lambda l: l.members([
                lambda m: m.comment(lambda s: s.text(" a")),
                lambda m: m.tspropertysignature(lambda p: p.key(a)),
                lambda m: m.comment(lambda s: s.block(True).text(" b ").trailing(True))]))))]))),
    lambda u: u.tstypealiasdeclaration(lambda s: s.id(lambda i: i.name("Y")).typeAnnotation(lambda t: t.tstypeliteral(lambda l: l.members([
        lambda m: m.tspropertysignature(lambda p: p.key(a)), lambda m: m.tsmethodsignature(lambda p: p.key(b))])))),
    lambda u: u.tstypealiasdeclaration(lambda s: s.id(lambda i: i.name("Z")).typeAnnotation(lambda t: t.tstypeliteral(lambda l: l))),
])
assert comments == """\
{ // only
}
{ /* only */
}
enum E { // a
    a,
    // b // c
}
namespace N {
    type X = {
        // a
        a; /* b */
    };
}
type Y = { a; b(); };
type Z = {};
"""
# A program's hashbang first; a trailing comment first in a program has no line to end
store.transaction(lambda t: t.uid("hashbang.ts").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"ws": R("Programs.Framework.Workspace"), "p": R("Programs.TypeScript5.Program")})
    .spec([lambda _: _.ws.files({"hashbang.ts": lambda e: e.file(_.p)}),
           lambda _: _.p.hashbang("/usr/bin/env node").body([lambda u: u.comment(lambda s: s.text(" a")), say_b])])]))]))
assert Print.print_module(store, "hashbang.ts") == "#!/usr/bin/env node\n// a\nb;\n"
assert printed("first.ts", [lambda u: u.comment(lambda s: s.text(" first").trailing(True))]) == "// first\n"

<a id="TS5PRT-08"></a>
## TS5PRT-08 · Reading the store: transactions, visitors, order, clauses and members alone, and every kind

`print_module` reads within the transaction open on the store, or within one of its own, and refuses a path the
workspace holds no file at; `print_file` prints a program's visitor and `print_statement` a statement's, a clause's or
a member's record, the one `Terms.kind_of` reads of an element of the container that holds it, each within the
transaction that read it. Statements print in their container's order, the order of its index, whatever the order
their elements were written in. A clause or a member alone prints at the level of none, a block within its method's
value too. A record of no kind the printer prints, a program or a part, is refused. The printer prints every kind.

In [ ]:
def held(prop):
    """What a property holds, as `Objects.held` reads it: a container's `Listed`, whose `values` are in its order."""
    found = []
    prop.as_value(lambda a: found.append(Objects.held(a)) or a)
    return found[0]


build("order.ts", {  # by index, written out of order
    2: lambda u: u.trystatement(lambda s: s
        .handler(lambda h: h.catchclause(lambda w: w.param(lambda p: p.identifier(lambda i: i.name("e")))
                                         .body(lambda k: k.body([say_b]))))
        .block(lambda k: k.body([say_b]))),
    1: lambda u: u.switchstatement(lambda s: s.discriminant(a).cases([lambda k: k.switchcase(lambda w: w.consequent([say_b]))])),
    3: lambda u: u.classdeclaration(lambda k: k.id(lambda i: i.name("C")).body(lambda y: y.body([
        lambda m: m.staticblock(lambda s: s.body([say_b])),
        lambda m: m.methoddefinition(lambda d: d.methodKind("method").key(a).value(lambda f: f.functionexpression(
            lambda f: f.block(lambda k: k.blockstatement(lambda k: k.body([lambda u: u.returnstatement(lambda s: s.argument(
                lambda e: e.arrowfunctionexpression(lambda r: r.block(lambda k: k.blockstatement(lambda k: k.body([say_c]))))))])))))),
        lambda m: m.tsindexsignature(lambda s: s.parameters([lambda p: p.identifier(lambda i: i.name("k"))]))]))),
    0: lambda u: u.variabledeclaration(lambda s: s.declarationKind("let").declarations([lambda d: d.id(a)])),
})
statements = ["let a;", "switch (a) {\n    default:\n        b;\n}", "try {\n    b;\n} catch (e) {\n    b;\n}",
              "class C {\n    static {\n        b;\n    }\n    a() {\n        return () => {\n            c;\n        };\n    }\n"
              "    [k];\n}"]
expected = "".join(statement + "\n" for statement in statements)
assert Print.print_module(store, "order.ts") == expected  # in a transaction of its own
with store.transaction():
    with raises(ValueError, match="transactions do not nest"):  # print_module opens its own: called with none open
        Print.print_module(store, "order.ts")
    program = store.singleton(Model.WORKSPACE).files({"path": "order.ts"})
    assert Print.print_file(program) == expected
    records = [Terms.kind_of(element) for element in held(program.body).values]
    assert [tag for tag, _ in records] == ["variabledeclaration", "switchstatement", "trystatement", "classdeclaration"]
    assert [Print.print_statement(record) for _, record in records] == statements
    # A clause and a member alone, at the level of none
    (case,) = [Terms.kind_of(element)[1] for element in held(records[1][1].cases).values]
    assert Print.print_statement(case) == "default:\n    b;"
    assert Print.print_statement(Terms.kind_of(held(records[2][1].handler))[1]) == "catch (e) {\n    b;\n}"
    members = [Terms.kind_of(element)[1] for element in held(held(records[3][1].body).body).values]
    assert [Print.print_statement(member) for member in members] == [
        "static {\n    b;\n}", "a() {\n    return () => {\n        c;\n    };\n}", "[k];"]
    # What is no program, statement, clause or member is refused
    with raises(TypeError, match="not a Programs.TypeScript5.Program"):
        Print.print_file(records[0][1])
    with raises(TypeError, match="no printer for the statement kind 'Programs.TypeScript5.Program'"):
        Print.print_statement(program)
    with raises(TypeError, match="no printer for the statement kind 'Programs.TypeScript5.VariableDeclarator'"):
        Print.print_statement(held(records[0][1].declarations).values[0])
with raises(LookupError, match="'nope.ts'"):
    Print.print_module(store, "nope.ts")
build("empty.ts", [])
assert Print.print_module(store, "empty.ts") == ""
# Every kind prints: the statements, clauses and members, the language's Error, and `TypeScript Types`' members
assert sorted(Print.KINDS) == sorted((*Statements.KINDS, "Error", "TSIndexSignature", "TSPropertySignature",
                                      "TSMethodSignature", "TSCallSignatureDeclaration", "TSConstructSignatureDeclaration"))

<a id="TS5PRT-09"></a>
## TS5PRT-09 · Invalid syntax prints as written

An `Error` where a statement stands prints its text, its first line at its level and the others as the source
indented them, and not its `body`, the statements recovered within it, which its text holds; an `Error` with no text,
nothing; an `error` term its text and a `missing` term nothing, as `Text.Printer` writes them. An `Error` where a part
of a statement stands prints its text in the part's place: a `for`'s initializer and left side, a default export's
declaration, an import's specifier, a module's name, an import-equals' reference, a `switch`'s case, a `try`'s
`catch` clause (on the line that closes its block, and nothing with no text), a class member, an enum member.

In [ ]:
m = lambda l: l.text("'m'")  # noqa: E731
invalid = printed("invalid.ts", [
    lambda u: u.error(lambda s: s.text("let = = 1;")),
    lambda u: u.functiondeclaration(lambda f: f.id(lambda i: i.name("f")).body(lambda k: k.body([
        lambda u: u.error(lambda s: s.text("if (a +) {\n        b;\n    }").body([say_b])),
        lambda u: u.error(lambda s: s),
        lambda u: u.returnstatement(lambda s: s.argument(lambda e: e.binaryexpression(lambda x: x
            .left(lambda t: t.error(lambda r: r.text("a <> b").terms([lambda t: t.typescript5(a), lambda t: t.typescript5(b)]))).operator("+")
            .right(lambda t: t.missing(lambda n: n.kind("identifier")))))),
    ]))),
    lambda u: u.forstatement(lambda s: s.init(lambda i: i.error(lambda r: r.text("let = 1"))).body(say_b)),
    lambda u: u.forofstatement(lambda s: s.left(lambda l: l.error(lambda r: r.text("const of"))).right(a).body(say_b)),
    lambda u: u.exportdefaultdeclaration(lambda s: s.declaration(lambda d: d.error(lambda r: r.text("class extends {}")))),
    lambda u: u.importdeclaration(lambda s: s.specifiers([lambda c_: c_.importdefaultspecifier(lambda p: p.local(lambda i: i.name("d"))),
                                                          lambda c_: c_.error(lambda r: r.text("* from"))]).source(m)),
    lambda u: u.tsmoduledeclaration(lambda s: s.moduleKind("namespace").id(lambda n: n.error(lambda r: r.text("1N")))
        .body(lambda y: y)),
    lambda u: u.tsimportequalsdeclaration(lambda s: s.id(lambda i: i.name("a")).moduleReference(lambda r: r.error(lambda r: r.text("require(")))),
    lambda u: u.switchstatement(lambda s: s.discriminant(a).cases([
        lambda k: k.switchcase(lambda w: w.test(b).consequent([say_b])),
        lambda k: k.error(lambda r: r.text("case:"))])),
    lambda u: u.trystatement(lambda s: s.block(lambda k: k.body([say_b])).handler(lambda h: h.error(lambda r: r.text("catch (a b) {}")))),
    lambda u: u.trystatement(lambda s: s.block(lambda k: k).handler(lambda h: h.error(lambda r: r)).finalizer(lambda k: k)),
    lambda u: u.classdeclaration(lambda k: k.id(lambda i: i.name("C")).body(lambda y: y.body([
        lambda m: m.error(lambda r: r.text("a b c;")), lambda m: m.propertydefinition(lambda p: p.key(a))]))),
    lambda u: u.tsenumdeclaration(lambda s: s.id(lambda i: i.name("E")).body(lambda y: y.members([
        lambda m: m.tsenummember(lambda e: e.id(a)), lambda m: m.error(lambda r: r.text("1 = 2"))]))),
])
assert invalid == """\
let = = 1;
function f() {
    if (a +) {
        b;
    }
    return a <> b + ;
}
for (let = 1;;)
    b;
for (const of of a)
    b;
export default class extends {}
import d, * from from 'm';
namespace 1N {}
import a = require(;
switch (a) {
    case b:
        b;
    case:
}
try {
    b;
} catch (a b) {}
try {} finally {}
class C {
    a b c;
    a;
}
enum E {
    a,
    1 = 2
}
"""